In [ ]:
#|default_exp _recipes

In [ ]:
#|hide
from nblite import nbl_export; nbl_export();

In [ ]:
#|export
import json
from pathlib import Path
from typing import Annotated, Optional

import typer

from netrun_cli._helpers import ConfigOpt, PrettyOpt, load_raw_data, output_json
from netrun.tools._helpers import get_recipes
from netrun.tools._recipes import execute_recipe, get_recipe_prompts

# Recipes Commands

List and run recipes defined in a netrun config.

In [ ]:
#|export
recipes_app = typer.Typer(help="List and run recipes.", no_args_is_help=True)


@recipes_app.command("list")
def recipes_list(
    config: ConfigOpt = None,
    pretty: PrettyOpt = True,
) -> None:
    """List available recipes."""
    raw, config_path = load_raw_data(config)
    recipes = get_recipes(raw)
    result = {name: r.model_dump() for name, r in recipes.items()}
    output_json(result, pretty)


@recipes_app.command("run")
def recipes_run(
    name: Annotated[str, typer.Argument(help="Recipe name to run.")],
    config: ConfigOpt = None,
    inputs: Annotated[Optional[str], typer.Option("--inputs", "-i", help="JSON string of input values.")] = None,
    output: Annotated[Optional[str], typer.Option("--output", "-o", help="Output file path for modified config.")] = None,
    pretty: PrettyOpt = True,
) -> None:
    """Run a recipe with JSON inputs."""
    raw, config_path = load_raw_data(config)
    recipes = get_recipes(raw)

    if name not in recipes:
        typer.echo(f"Error: recipe '{name}' not found.", err=True)
        raise typer.Exit(1)

    recipe_cfg = recipes[name]
    recipe_path = Path(recipe_cfg.path)
    if not recipe_path.is_absolute():
        recipe_path = config_path.parent / recipe_path

    if not recipe_path.exists():
        typer.echo(f"Error: recipe file not found: {recipe_path}", err=True)
        raise typer.Exit(1)

    # Parse inputs
    input_values: dict = {}
    if inputs:
        try:
            input_values = json.loads(inputs)
        except json.JSONDecodeError as e:
            typer.echo(f"Error: invalid JSON inputs: {e}", err=True)
            raise typer.Exit(1)

    # If no inputs given, check if recipe has prompts and show them
    if not inputs:
        prompts = get_recipe_prompts(recipe_path, raw)
        if prompts:
            typer.echo("Recipe requires inputs. Use --inputs with JSON:", err=True)
            for p in prompts:
                info = f"  {p.name} ({p.type}): {p.label}"
                if p.default is not None:
                    info += f" [default: {p.default}]"
                if p.options:
                    info += f" options: {p.options}"
                typer.echo(info, err=True)
            raise typer.Exit(1)

    try:
        modified = execute_recipe(recipe_path, raw, input_values)
    except Exception as e:
        typer.echo(f"Error running recipe: {e}", err=True)
        raise typer.Exit(1)

    if output:
        Path(output).write_text(json.dumps(modified, indent=2, default=str))
        typer.echo(f"Written to {output}")
    else:
        output_json(modified, pretty)